# Notebook 09 — Walk-Forward Out-of-Sample Validation

**Adaptive Pair Trading | Ayush Arora (MQMS2404)**

---

## Why Walk-Forward Validation?

A single train/test split is a weak test. Any backtest result could be specific
to the exact split point chosen. Walk-forward validation solves this by:

1. Dividing the 10-year history into **rolling 2-year training + 6-month test windows**
2. Re-estimating the hedge ratio, Z-score parameters, and strategy on each training window
3. Measuring performance **only on the held-out test period** for each window
4. Aggregating ~14 out-of-sample test periods to get a **distribution of OOS Sharpe ratios**

A strategy is credible only if its OOS Sharpe distribution is:
- Centred above zero (genuine edge, not overfitting)
- Consistent across time periods (not a one-decade fluke)
- Statistically significant (t-test on OOS Sharpe > 0)

This notebook uses the **TATASTEEL / HINDALCO** pair (from NB00/02) as the primary
validation case, then generalises to the top pairs from NB08.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from dateutil.relativedelta import relativedelta
from scipy import stats
import warnings
from config import (
    PRICES_FILE, TOP_PAIRS_FILE,
    ROLL_WINDOW, ROLL_MIN_PERIODS,
    ENTRY_Z, EXIT_Z, TC,
    WF_TRAIN_MONTHS, WF_TEST_MONTHS,
    KF_DELTA, KF_INIT_P
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

In [ ]:
prices = pd.read_csv(PRICES_FILE, index_col=0, parse_dates=True)

# Primary validation pair
A_COL = 'TATASTEEL.NS'
B_COL = 'HINDALCO.NS'
A = prices[A_COL]
B = prices[B_COL]

print(f'Price data: {prices.shape}')
print(f'Validation pair: {A_COL} / {B_COL}')
print(f'Walk-forward params: {WF_TRAIN_MONTHS}m train / {WF_TEST_MONTHS}m test')
print(f'Strategy: entry ±{ENTRY_Z}σ, exit ±{EXIT_Z}σ, TC {TC*10000:.0f}bps')

## Walk-Forward Engine

The engine loops through rolling windows. For each window it:
1. **Fits OLS** on training data → gets β and α
2. **Builds spread** on test data using those fitted parameters
3. **Computes rolling Z-score** on test data (using training period's spread statistics as seed)
4. **Runs the strategy** on the test period only
5. **Records** Sharpe ratio, max drawdown, number of trades, hit rate

All strategy parameters (ENTRY_Z, EXIT_Z, TC) are **fixed** — no optimisation happens
in each window. This rules out in-sample parameter overfitting.

In [ ]:
def ols_spread(a, b):
    model = sm.OLS(a, sm.add_constant(b)).fit()
    beta  = model.params.iloc[1]
    alpha = model.params.iloc[0]
    return a - beta * b - alpha, beta, alpha

def run_strategy(zscore, spread, tc=TC):
    """Baseline Z-score strategy. Returns daily % P&L series."""
    spread_mean_abs = spread.dropna().abs().mean()
    spread_ret_pct  = spread.diff() / (spread_mean_abs + 1e-10)
    pos  = pd.Series(0.0, index=zscore.index)
    curr = 0.0
    for t in zscore.index:
        z = zscore.loc[t]
        if pd.isna(z):
            pos.loc[t] = 0.0
            continue
        if curr == 0:
            if   z >  ENTRY_Z: curr = -1.0
            elif z < -ENTRY_Z: curr =  1.0
        elif curr ==  1 and z > -EXIT_Z: curr = 0.0
        elif curr == -1 and z <  EXIT_Z: curr = 0.0
        pos.loc[t] = curr
    trade = pos.diff().abs().fillna(0)
    pnl   = pos.shift(1) * spread_ret_pct - trade * tc
    return pnl.fillna(0), pos

def sharpe(pnl):
    r = pnl[pnl != 0]
    if len(r) < 20:
        return np.nan
    return r.mean() / r.std() * np.sqrt(252)

def max_dd(pnl):
    cum  = (1 + pnl).cumprod()
    peak = cum.cummax()
    return ((cum - peak) / peak).min() * 100

def hit_rate(pnl):
    r = pnl[pnl != 0]
    return (r > 0).mean() * 100 if len(r) > 0 else np.nan

print('Walk-forward engine defined.')

In [ ]:
def build_windows(index, train_months, test_months):
    """Generate (train_start, train_end, test_start, test_end) tuples."""
    windows = []
    start   = index[0]
    end     = index[-1]
    while True:
        train_end  = start + relativedelta(months=train_months) - relativedelta(days=1)
        test_start = train_end + relativedelta(days=1)
        test_end   = test_start + relativedelta(months=test_months) - relativedelta(days=1)
        if test_end > end:
            break
        windows.append((start, train_end, test_start, test_end))
        start = start + relativedelta(months=test_months)   # roll by test window
    return windows

windows = build_windows(prices.index, WF_TRAIN_MONTHS, WF_TEST_MONTHS)
print(f'Walk-forward windows: {len(windows)}')
for i, (ts, te, ss, se) in enumerate(windows, 1):
    print(f'  Window {i:2d}: Train {ts.date()} → {te.date()} | Test {ss.date()} → {se.date()}')

In [ ]:
print('Running walk-forward backtest...')
wf_results = []
all_oos_pnl = []

for i, (train_s, train_e, test_s, test_e) in enumerate(windows, 1):
    # ── Training period: fit OLS ──────────────────────────────────────────
    a_train = A.loc[train_s:train_e].dropna()
    b_train = B.loc[train_s:train_e].dropna()
    if len(a_train) < 60:
        continue
    spread_train, beta, alpha = ols_spread(a_train, b_train)
    train_mean = spread_train.mean()
    train_std  = spread_train.std()

    # ── Test period: apply fitted parameters ─────────────────────────────
    a_test = A.loc[test_s:test_e].dropna()
    b_test = B.loc[test_s:test_e].dropna()
    if len(a_test) < 20:
        continue
    spread_test = a_test - beta * b_test - alpha
    # Z-score using training-period statistics (no look-ahead)
    zscore_test = (spread_test - train_mean) / (train_std + 1e-10)

    pnl_test, pos_test = run_strategy(zscore_test, spread_test)
    trades = int((pos_test.diff().abs() > 0).sum() // 2)

    result = {
        'Window': i,
        'Train Start': train_s.date(),
        'Train End':   train_e.date(),
        'Test Start':  test_s.date(),
        'Test End':    test_e.date(),
        'Beta (OLS)':  round(beta, 4),
        'OOS Sharpe':  round(sharpe(pnl_test), 3),
        'OOS Max DD%': round(max_dd(pnl_test), 2),
        'Hit Rate%':   round(hit_rate(pnl_test), 1),
        'Trades':      trades,
    }
    wf_results.append(result)
    all_oos_pnl.append(pnl_test)

wf_df = pd.DataFrame(wf_results)
print(f'Completed {len(wf_df)} windows')
wf_df

## OOS Performance Analysis

In [ ]:
oos_sharpes = wf_df['OOS Sharpe'].dropna()

# t-test: is mean OOS Sharpe significantly different from zero?
t_stat, p_val = stats.ttest_1samp(oos_sharpes, 0)

print('=== OUT-OF-SAMPLE PERFORMANCE SUMMARY ===')
print(f'Windows tested       : {len(oos_sharpes)}')
print(f'Mean OOS Sharpe      : {oos_sharpes.mean():+.3f}')
print(f'Median OOS Sharpe    : {oos_sharpes.median():+.3f}')
print(f'Std OOS Sharpe       : {oos_sharpes.std():.3f}')
print(f'Positive windows     : {(oos_sharpes > 0).sum()} / {len(oos_sharpes)} ({(oos_sharpes > 0).mean()*100:.0f}%)')
print(f'Sharpe > 0.5 windows : {(oos_sharpes > 0.5).sum()} / {len(oos_sharpes)}')
print(f't-statistic          : {t_stat:.3f}')
print(f'p-value (H0: μ=0)    : {p_val:.4f}  → {"Significant" if p_val < 0.05 else "Not significant"} at 5%')
print(f'\nOLS Beta stability:')
print(f'  Min β: {wf_df["Beta (OLS)"].min():.4f}  Max β: {wf_df["Beta (OLS)"].max():.4f}  Std: {wf_df["Beta (OLS)"].std():.4f}')

In [ ]:
# Concatenate all OOS P&L into one continuous series
if all_oos_pnl:
    oos_combined = pd.concat(all_oos_pnl)
    oos_cum = (1 + oos_combined).cumprod() - 1
else:
    oos_combined = pd.Series(dtype=float)
    oos_cum = pd.Series(dtype=float)

fig = plt.figure(figsize=(14, 11))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)

# ── OOS equity curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
if len(oos_cum) > 0:
    ax1.plot(oos_cum * 100, color='crimson', linewidth=1.2)
    ax1.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax1.fill_between(oos_cum.index, oos_cum * 100, 0,
                     where=(oos_cum >= 0), alpha=0.1, color='green')
    ax1.fill_between(oos_cum.index, oos_cum * 100, 0,
                     where=(oos_cum < 0), alpha=0.1, color='red')
ax1.set_title('Concatenated Out-of-Sample Equity Curve (all windows stitched)', fontweight='bold')
ax1.set_ylabel('Cumulative Return (%)')
ax1.grid(alpha=0.3)

# ── Sharpe by window ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
colors = ['green' if s > 0 else 'red' for s in wf_df['OOS Sharpe']]
ax2.bar(wf_df['Window'], wf_df['OOS Sharpe'], color=colors, alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axhline(oos_sharpes.mean(), color='navy', linestyle='--',
            linewidth=1.2, label=f'Mean={oos_sharpes.mean():.3f}')
ax2.set_title('OOS Sharpe Ratio by Window')
ax2.set_xlabel('Window #')
ax2.set_ylabel('Sharpe Ratio')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)

# ── Sharpe distribution ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(oos_sharpes, bins=min(10, len(oos_sharpes)),
         color='steelblue', edgecolor='white', alpha=0.8)
ax3.axvline(0, color='red', linewidth=1.5, linestyle='--', label='Zero')
ax3.axvline(oos_sharpes.mean(), color='navy', linewidth=1.5,
            label=f'Mean={oos_sharpes.mean():.3f}')
ax3.set_title('OOS Sharpe Distribution')
ax3.set_xlabel('Sharpe Ratio')
ax3.set_ylabel('Frequency')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.2)

# ── Rolling OLS beta ──────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(range(1, len(wf_df)+1), wf_df['Beta (OLS)'],
         marker='o', color='darkorange', linewidth=1.2, markersize=4)
ax4.axhline(wf_df['Beta (OLS)'].mean(), color='navy', linestyle='--', linewidth=1)
ax4.set_title('OLS Hedge Ratio (β) Stability')
ax4.set_xlabel('Window #')
ax4.set_ylabel('β')
ax4.grid(alpha=0.2)

# ── Drawdown by window ────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.bar(wf_df['Window'], wf_df['OOS Max DD%'].abs(),
        color='darkred', alpha=0.6)
ax5.set_title('OOS Max Drawdown by Window')
ax5.set_xlabel('Window #')
ax5.set_ylabel('Max Drawdown (%)')
ax5.grid(alpha=0.2)

plt.savefig('walk_forward_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── In-sample vs out-of-sample Sharpe comparison ─────────────────────────
is_sharpes = []
for train_s, train_e, test_s, test_e in windows:
    a_tr = A.loc[train_s:train_e].dropna()
    b_tr = B.loc[train_s:train_e].dropna()
    if len(a_tr) < 60:
        continue
    spread_tr, beta, alpha = ols_spread(a_tr, b_tr)
    roll_m = spread_tr.rolling(60, min_periods=30).mean()
    roll_s = spread_tr.rolling(60, min_periods=30).std()
    zscore_tr = (spread_tr - roll_m) / (roll_s + 1e-10)
    pnl_tr, _ = run_strategy(zscore_tr, spread_tr)
    is_sharpes.append(sharpe(pnl_tr))

print('=== IN-SAMPLE vs OUT-OF-SAMPLE SHARPE ===')
print(f'{"Window":>8} {"IS Sharpe":>12} {"OOS Sharpe":>12} {"Gap":>8}')
for i, (s_is, s_oos) in enumerate(zip(is_sharpes, wf_df['OOS Sharpe']), 1):
    gap = s_oos - s_is if not np.isnan(s_is) else np.nan
    print(f'{i:>8} {s_is:>12.3f} {s_oos:>12.3f} {gap:>8.3f}')
print(f'\n{"Mean":>8} {np.nanmean(is_sharpes):>12.3f} {oos_sharpes.mean():>12.3f} {oos_sharpes.mean()-np.nanmean(is_sharpes):>8.3f}')
print(f'\nIS > OOS (overfitting signal): {np.nanmean(is_sharpes) > oos_sharpes.mean()}')

## Conclusion

### Interpretation guide

| Result | Interpretation |
|--------|---------------|
| Mean OOS Sharpe > 0 & p < 0.05 | Statistically significant edge — strategy has genuine predictive power |
| Mean OOS Sharpe > 0 & p > 0.05 | Positive but not yet significant — more data or pairs needed |
| Mean OOS Sharpe < 0 | No edge — strategy needs redesign |
| IS Sharpe ≈ OOS Sharpe | Low overfitting — fixed parameters generalise well |
| IS Sharpe >> OOS Sharpe | Overfitting — parameters optimised in-sample, fail OOS |

### Key design safeguards
- **No parameter optimisation per window** — ENTRY_Z, EXIT_Z, TC are fixed constants from `config.py`
- **Training statistics for test Z-score** — test period normalises using training-period μ/σ, not its own
- **Rolling windows prevent look-ahead** — the hedge ratio β estimated for each period only uses past data

This notebook provides the academic rigour to defend the strategy's validity against
the standard critique: *"any backtest can be made to look good in-sample."*